# CosMx spatial proteomics: CD section, RNA + protein processing

Builds the CD object used by `05_protein_cell_types_myeloid_treg.py`:

1. reads the RNA and protein flat files of the CD section
2. RNA QC and total-count normalisation; RNA AUCell labels from `02_cd_rna_aucell_annotation.R` (cycling cells removed)
3. protein matrix matched to the QC'd cells, log1p and centred on each cell's geometric mean (`<protein>_geo` columns)
4. writes `rna_norm_pr_geonorm_upd_anno.h5ad`

In [ ]:
# ── Paths ──
UC_DIR     = "/path/to/imputation_model/data/Spatial_Protein_CosMx"
CD_DIR     = "/path/to/cosmx_data/cd_multiomics"
OUTPUT_DIR = UC_DIR + "/figures"

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import os
import scipy.stats as stats
from scipy.sparse import csr_matrix
import anndata as adfrom matplotlib.colors import Normalize
from scipy.spatial import cKDTree
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import matplotlib.cm as cm


In [ ]:
load_PATH = "/path/to/cosmx_data/cd_multiomics/Cos4_CD_M_batch1"

#Cos4_303i_312i
#Cos4_CD_M_batch1

In [ ]:
#rna='Cos4_303i_312i_RNA_2_7.26'

rna='Cos4_CD_M_batch1_RNA_1_7.26'

In [ ]:
#sample='303i_312i'
sample='CD_M_batch1'

In [ ]:
# === RNA ===

rna_path = os.path.join(load_PATH, rna, "Updated/FlatFiles")

print("Loading RNA expression matrix...")
rna_expr = pd.read_csv(rna_path+'/'+sample+"_exprMat_file.csv.gz")
rna_meta = pd.read_csv(rna_path+'/'+sample+"_metadata_file.csv.gz")
rna_expr=rna_expr.drop('fov',axis=1)

cell_id_col = [c for c in rna_expr.columns if 'cell' in c.lower()][0]
rna_cells = rna_expr[cell_id_col].values
gene_cols = [c for c in rna_expr.columns if c != cell_id_col]

# Drop negprobes, falsecodes, system controls
drop_prefixes = ('NegPrb', 'NegControl', 'SystemControl', 'FalseCode', 'Falsecode')
real_gene_cols = [c for c in gene_cols if not c.startswith(drop_prefixes)]

print(f"RNA: {len(rna_cells)} cells, {len(real_gene_cols)} genes (dropped {len(gene_cols) - len(real_gene_cols)} controls)")


In [ ]:

X_rna = csr_matrix(rna_expr[real_gene_cols].values)
adata_rna = ad.AnnData(X=X_rna)
adata_rna.obs_names = [str(c) for c in rna_cells]
adata_rna.var_names = real_gene_cols


In [ ]:
for col in rna_meta.columns:
    adata_rna.obs[col] = rna_meta[col].values

In [ ]:
pr='Cos4_CD_M_batch1_PR_1_7.22'
#Cos4_303i_312i_PR_2_7.22

In [ ]:
pr_pfix='Cos4CD_M1CD_M2PR1722'
#Cos4303i312iPR2722

In [ ]:
pr_path = os.path.join(load_PATH,pr, "FlatFiles")

print("\nLoading Protein expression matrix...")
pr_expr = pd.read_csv(pr_path+'/'+pr_pfix+"_exprMat_file.csv.gz")
pr_meta = pd.read_csv(pr_path+'/'+pr_pfix+"_metadata_file.csv.gz")
pr_expr=pr_expr.drop('fov',axis=1)
cell_id_col_pr = [c for c in pr_expr.columns if 'cell' in c.lower()][0]
pr_cells = pr_expr[cell_id_col_pr].values
protein_cols = [c for c in pr_expr.columns if c != cell_id_col_pr]


In [ ]:
neg_cols_pr = [c for c in protein_cols if 'neg' in c.lower() or 'control' in c.lower()]
real_protein_cols = [c for c in protein_cols if c not in neg_cols_pr]

print(f"Protein: {len(pr_cells)} cells, {len(real_protein_cols)} proteins (dropped {len(neg_cols_pr)} controls)")

X_pr = csr_matrix(pr_expr[real_protein_cols].values.astype(np.float32))
adata_pr = ad.AnnData(X=X_pr)
adata_pr.obs_names = [str(c) for c in pr_cells]
adata_pr.var_names = real_protein_cols


In [ ]:

for col in pr_meta.columns:
    adata_pr.obs[col] = pr_meta[col].values

In [ ]:
adata_pr_filter = adata_pr[adata_pr.obs['cell'].isin(adata_rna.obs['cell'])]

In [ ]:
adata_rna_qc = adata_rna.copy()

In [ ]:
sc.pp.calculate_qc_metrics(
    adata_rna_qc, inplace=True, log1p=True
)

In [ ]:
sc.pl.violin(
    adata_rna_qc,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(adata_rna_qc, min_counts=200)
sc.pp.filter_cells(adata_rna_qc, min_genes=100)
sc.pp.filter_cells(adata_rna_qc, max_counts=3500)
sc.pp.filter_cells(adata_rna_qc, max_genes=2000)
adata_rna_qc=adata_rna_qc[adata_rna_qc.obs['Area']<=30000]

In [ ]:
sc.pl.violin(
    adata_rna_qc,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
adata_rna_qc.layers['raw']=adata_rna_qc.X

In [ ]:
sc.pp.normalize_total(adata_rna_qc, inplace=True)

In [ ]:
rna1_df=pd.DataFrame(adata_rna_qc.X.toarray())


In [ ]:
rna1_df.index = adata_rna_qc.obs['cell_id'].tolist()
rna1_df.columns = adata_rna_qc.var_names.tolist()


In [ ]:
rna1_df.to_csv(load_PATH+'/UPD_norm_nolog_rna.csv')

In [ ]:
rna1_anno = pd.read_csv(load_PATH+'/aucell_anno_upd.csv')

In [ ]:
mapper= pd.read_csv('/path/to/scrna/cd/cell_type_category_map.csv')

In [ ]:
# 1. Clean the names
mapper['cell_type_aucell'] = mapper['cell type'].str.replace(r'[+\-\/()]', ' ', regex=True)
mapper['cell_type_aucell'] = mapper['cell_type_aucell'].str.replace(r'\s+', ' ', regex=True).str.strip()

# 2. Disambiguate duplicates by appending " 1", " 2", etc.
mapper['cell_type_aucell'] = (
    mapper.groupby('cell_type_aucell').cumcount().astype(str).replace('0', '', regex=False)
    .radd(mapper['cell_type_aucell'] + ' ').str.strip()
)

In [ ]:
rna1_anno['label_clean'] = rna1_anno['label'].str.rstrip()

In [ ]:
rna1_anno_map=rna1_anno.merge(mapper, how = 'left', left_on = 'label_clean',right_on = 'cell_type_aucell')

In [ ]:
adata_rna_qc.obs['cell_type'] = rna1_anno_map['cell type'].tolist()
adata_rna_qc.obs['cell_type_short'] = rna1_anno_map['cell type short'].tolist()
adata_rna_qc.obs['cell_category'] = rna1_anno_map['cell category'].tolist()
adata_rna_qc.obs['cell_type_general'] = rna1_anno_map['cell type general'].tolist()

In [ ]:
adata_rna_qc.obsm['spatial_fov']=adata_rna_qc.obs[['CenterX_global_px', 'CenterY_global_px']].values.astype(int)

In [ ]:
adata_rna_qc=adata_rna_qc[adata_rna_qc.obs['cell_category']!='Cycling']

In [ ]:
adata_rna_qc_312 = adata_rna_qc[(adata_rna_qc.obs['fov'] <=29)].copy()
adata_rna_qc_312

In [ ]:
adata_rna_qc_375 = adata_rna_qc[(adata_rna_qc.obs['fov'] <=29)].copy()
adata_rna_qc_375

In [ ]:
adata_pr_filter = adata_pr_filter[adata_pr_filter.obs['cell_id'].isin(adata_rna_qc.obs['cell_id'].tolist())]
adata_pr_filter

In [ ]:
adata_pr_filter_geo=adata_pr_filter.copy()

In [ ]:
# 1. Log-transform if necessary (for count data)
sc.pp.log1p(adata_pr_filter_geo)

# 2. Compute the geometric mean for each cell (margin=1, along rows)
geometric_mean = np.exp(np.mean(np.log(adata_pr_filter_geo.X.toarray() + 1), axis=1))

# 3. Normalize by dividing each value by the geometric mean (center each cell)
adata_pr_filter_geo.X = adata_pr_filter_geo.X / geometric_mean[:, np.newaxis]


In [ ]:
pro1_geo = pd.DataFrame(adata_pr_filter_geo.X.toarray())
pro1_geo.index = adata_pr_filter_geo.obs['cell_id'].tolist()
pro1_geo.columns = adata_pr_filter_geo.var_names.tolist()

In [ ]:
pro1_geo.columns = [i+'_geo' for i in pro1_geo.columns.tolist()]

In [ ]:
adata_rna_qc.obs = adata_rna_qc.obs.merge(pro1_geo, left_on = 'cell_id', right_index=True)

In [ ]:
adata_rna_qc_312 = adata_rna_qc[(adata_rna_qc.obs['fov'] <=29)].copy()
adata_rna_qc_312

In [ ]:
adata_rna_qc_375 = adata_rna_qc[(adata_rna_qc.obs['fov'] <=29)].copy()
adata_rna_qc_375

In [ ]:
rna2_pr = adata_rna_qc_375.obs[['cell_id','4-1BB_geo', 'B7-H3_geo', 'Bcl-2_geo', 'Beta-catenin_geo', 'CCR7_geo', 'CD11b_geo', 'CD11c_geo', 'CD123_geo', 'CD127_geo', 'CD138_geo', 'CD14_geo', 'CD15_geo', 'CD16_geo', 'CD163_geo', 'CD19_geo', 'CD20_geo', 'CD27_geo', 'CD3_geo', 'CD31_geo', 'CD34_geo', 'CD38_geo', 'CD39_geo', 'CD4_geo', 'CD40_geo', 'CD45_geo', 'CD45RA_geo', 'CD56_geo', 'CD68_geo', 'CD8_geo', 'CTLA4_geo', 'Channel-CD3_geo', 'Channel-CD45_geo', 'Channel-DNA_geo', 'Channel-Membrane_geo', 'Channel-PanCK_geo', 'EGFR_geo', 'EpCAM_geo', 'FABP4_geo', 'FOXP3_geo', 'Fibronectin_geo', 'GITR_geo', 'GZMA_geo', 'GZMB_geo', 'HLA-DR_geo', 'Her2_geo', 'ICAM1_geo', 'ICOS_geo', 'IDO1_geo', 'IL-18_geo', 'IL-1b_geo', 'IgD_geo', 'Ki-67_geo', 'LAG3_geo', 'LAMP1_geo', 'NF-kB p65_geo', 'PD-1_geo', 'PD-L1_geo', 'PD-L2_geo', 'SMA_geo', 'STING_geo', 'TCF7_geo', 'Tim-3_geo', 'VISTA_geo', 'Vimentin_geo', 'iNOS_geo', 'p53_geo', 'pan-RAS_geo', 'Ms IgG1_geo', 'Rb IgG_geo','cell_category']]
rna2_pr.index = rna2_pr['cell_id'].tolist()
rna2_pr=rna2_pr.drop('cell_id',axis=1)
rna2_pr

In [ ]:
adata_rna_qc.write_h5ad(f"{load_PATH}/rna_norm_pr_geonorm_not_upd_anno.h5ad")

In [ ]:
adata_rna_qc.write_h5ad(f"{load_PATH}/rna_norm_pr_geonorm_upd_anno.h5ad")